In [1]:
import pandas as pd
import numpy as np
np.set_printoptions(precision=6, suppress=True)

import matplotlib.pyplot as plt
from tqdm import tqdm

import tensorflow as tf
import tensorflow_addons as tfa
from tensorflow.keras import *
tf.__version__

'2.3.0'

In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        print(e)

2 Physical GPUs, 2 Logical GPUs


In [3]:
strategy = tf.distribute.MirroredStrategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


# Hyperparameters

In [4]:
BEST_PATH = './models/ResNet_1D_v3.h5'
TRAINING_EPOCHS = 200
LEARNING_RATE = 0.0015
EPSILON = 1e-05
BATCH_SIZE = 256

# Data loading

In [5]:
l = np.load('./data/solar_dataset.npz', allow_pickle=True)
train_input = l['train_input']
train_label = l['train_label']
val_input = l['val_input']
val_label = l['val_label']
test_input = l['test_input']
MAXS = l['MAXS']
MINS = l['MINS']

In [6]:
train_input = train_input.astype('float32')
train_label = train_label.astype('float32')
val_input = val_input.astype('float32')
val_label = val_label.astype('float32')
test_input = test_input.astype('float32')

In [7]:
print(train_input.shape)
print(train_label.shape)

(39528, 336, 11)
(39528, 96, 1)


In [8]:
print(val_input.shape)
print(val_label.shape)

(4392, 336, 11)
(4392, 96, 1)


In [9]:
print(MAXS)
print(MINS)

[ 1440.         528.        1059.          12.         100.
    35.          36.376067  1746.051202  7541.       24697.
    99.913939]
[ 30.         0.         0.         0.         7.59     -19.
   1.958976   0.         0.         0.         0.      ]


In [10]:
with strategy.scope():
    train_dataset = tf.data.Dataset.from_tensor_slices((train_input, train_label))
    train_dataset = train_dataset.cache().shuffle(20000).padded_batch(BATCH_SIZE)
    val_dataset = tf.data.Dataset.from_tensor_slices((val_input, val_label))
    val_dataset = val_dataset.cache().shuffle(20000).padded_batch(BATCH_SIZE)

# Model construction

In [11]:
class ConvBlock(layers.Layer):
    def __init__(self, filters, kernel_size):
        super(ConvBlock, self).__init__()
        self.f = filters
        self.k = kernel_size
        
        self.downsampling = layers.Conv1D(self.f, 1, kernel_initializer='glorot_normal', padding='same')
        self.downbatch = tfa.layers.InstanceNormalization()
    
        self.conv1 = layers.Conv1D(self.f/4, 1, kernel_initializer='glorot_normal', padding='same')
        self.batch1 = tfa.layers.InstanceNormalization()
        self.activation1 = layers.Activation(tf.nn.relu)
        
        self.conv2_1 = layers.Conv1D(self.f/4, self.k[0], kernel_initializer='glorot_normal', padding='same')
        self.conv2_2 = layers.Conv1D(self.f/4, self.k[1], kernel_initializer='glorot_normal', padding='same')
        self.conv2_3 = layers.Conv1D(self.f/4, self.k[2], kernel_initializer='glorot_normal', padding='same')
        self.conv2_4 = layers.Conv1D(self.f/4, self.k[3], kernel_initializer='glorot_normal', padding='same')
        self.conv2_5 = layers.Conv1D(self.f/4, self.k[4], kernel_initializer='glorot_normal', padding='same')
        self.batch2 = tfa.layers.InstanceNormalization()
        self.activation2 = layers.Activation(tf.nn.relu)
        
        self.outconv = layers.Conv1D(self.f, 1, kernel_initializer='glorot_normal', padding='same')
        self.outbatch = tfa.layers.InstanceNormalization()
        self.outact = layers.Activation(tf.nn.relu)
        
    def call(self, inp):
        
        shortcut = self.downbatch(self.downsampling(inp))
        
        inp = self.activation1(self.batch1(self.conv1(inp)))
        
        inp = self.conv2_1(inp) + self.conv2_2(inp) + self.conv2_3(inp) + self.conv2_4(inp) + self.conv2_5(inp)
        inp = self.activation2(self.batch2(inp))
        
        inp = self.outbatch(self.outconv(inp))
        inp = self.outact(layers.add([shortcut, inp]))
                
        return inp

In [12]:
class ResNet1D(Model):
    def __init__(self):
        super(ResNet1D, self).__init__()
        self.n = [64, 128, 128, 256, 256] # number of nodes
        self.k = [1, 3, 5, 10, 20] # kernal size
        self.s = 2 # stride (= pooling size)
        self.p = tf.constant([[0, 0], [12, 12], [0, 0]])
        
        self.sc_gate = layers.Dense(self.n[4], activation=tf.nn.sigmoid)
        
        self.conv_block1_1 = ConvBlock(self.n[0], self.k)
        self.conv_block1_2 = ConvBlock(self.n[0], self.k)
        self.conv_block1_3 = ConvBlock(self.n[0], self.k)
        self.conv_block1_4 = ConvBlock(self.n[0], self.k)
        
        self.conv_block2_1 = ConvBlock(self.n[1], self.k)
        self.conv_block2_2 = ConvBlock(self.n[1], self.k)
        self.conv_block2_3 = ConvBlock(self.n[1], self.k)
        self.conv_block2_4 = ConvBlock(self.n[1], self.k)
        self.gate2 = layers.Dense(self.n[1], activation=tf.nn.sigmoid)
        
        self.conv_block3_1 = ConvBlock(self.n[2], self.k)
        self.conv_block3_2 = ConvBlock(self.n[2], self.k)
        self.conv_block3_3 = ConvBlock(self.n[2], self.k)
        self.conv_block3_4 = ConvBlock(self.n[2], self.k)
        self.gate3 = layers.Dense(self.n[2], activation=tf.nn.sigmoid)
        self.pool3 = layers.Conv1D(self.n[3], self.s, self.s)
        
        self.conv_block4_1 = ConvBlock(self.n[3], self.k)
        self.conv_block4_2 = ConvBlock(self.n[3], self.k)
        self.conv_block4_3 = ConvBlock(self.n[3], self.k)
        self.conv_block4_4 = ConvBlock(self.n[3], self.k)
        self.conv_block4_5 = ConvBlock(self.n[3], self.k)
        self.gate4 = layers.Dense(self.n[3], activation=tf.nn.sigmoid)
        self.pool4 = layers.Conv1D(self.n[4], self.s, self.s)
        
        self.conv_block5_1 = ConvBlock(self.n[4], self.k)
        self.conv_block5_2 = ConvBlock(self.n[4], self.k)
        self.conv_block5_3 = ConvBlock(self.n[4], self.k)
        self.conv_block5_4 = ConvBlock(self.n[4], self.k)
        self.conv_block5_5 = ConvBlock(self.n[4], self.k)
        self.conv_block5_6 = ConvBlock(self.n[4], self.k)
        self.gate5 = layers.Dense(self.n[4], activation=tf.nn.sigmoid)
        
        self.output_conv = layers.Conv1D(3, 1, activation='sigmoid')
        
    def call(self, inp):
        
        shortcut = inp[:, -96:, :]
        shortcut = self.sc_gate(shortcut)
        
        conv1 = self.conv_block1_1(inp)
        conv1 = self.conv_block1_2(conv1)
        conv1 = self.conv_block1_3(conv1)
        conv1 = self.conv_block1_4(conv1)
        
        conv2 = self.conv_block2_1(conv1)
        conv2 = self.conv_block2_2(conv2)
        conv2 = self.conv_block2_3(conv2)
        conv2 = self.conv_block2_4(conv2)
        conv2 = self.gate2(conv2)
        
        conv3 = self.conv_block3_1(conv2)
        conv3 = self.conv_block3_2(conv3)
        conv3 = self.conv_block3_3(conv3)
        conv3 = self.conv_block3_4(conv3)
        conv3 = self.gate3(conv3)
        conv3 = self.pool3(conv3)
        
        conv4 = self.conv_block4_1(conv3)
        conv4 = self.conv_block4_2(conv4)
        conv4 = self.conv_block4_3(conv4)
        conv4 = self.conv_block4_4(conv4)
        conv4 = self.conv_block4_5(conv4)
        conv4 = self.gate4(conv4)
        conv4 = self.pool4(tf.pad(conv4, self.p, 'REFLECT'))

        conv5 = self.conv_block5_1(conv4)
        conv5 = self.conv_block5_2(conv5)
        conv5 = self.conv_block5_3(conv5)
        conv5 = self.conv_block5_4(conv5)
        conv5 = self.conv_block5_5(conv5)
        conv5 = self.conv_block5_6(conv5)
        conv5 = self.gate5(conv5)
        
        conv5 = layers.concatenate([shortcut, conv5])
        
        return self.output_conv(conv5)

In [13]:
def loss_function(y_true, y_pred):
    loss = 0
    
    for i, p in zip([0, 1, 2], [0.1, 0.5, 0.9]):
        loss += tfa.losses.pinball_loss(y_true[..., 0], y_pred[..., i], p)
        
    return loss/3

In [14]:
# def loss_function(y_true, y_pred):
#     loss = 0
    
#     for i in range(9):
#         loss += tfa.losses.pinball_loss(y_true[..., 0], y_pred[..., i], (i+1)/10)
        
#     return loss/10

In [15]:
callbacks = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=.5, patience=2, verbose=0, mode='min',
    min_delta=0.0001, cooldown=0, min_lr=0)

save = tf.keras.callbacks.ModelCheckpoint(
    BEST_PATH, monitor='val_loss', verbose=0,
    save_best_only=True, save_weights_only=True, mode='min', save_freq='epoch')

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=0.0001, patience=5) 

In [16]:
with strategy.scope():
    opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, epsilon=EPSILON)
    model = ResNet1D()
    model.compile(optimizer=opt, loss=loss_function)
    model.fit(train_dataset, epochs=TRAINING_EPOCHS, validation_data=val_dataset,
                  verbose=1, callbacks=[callbacks, save, early_stop]) 

Epoch 1/200
Instructions for updating:
Use `tf.data.Iterator.get_next_as_optional()` instead.
INFO:tensorflow:batch_all_reduce: 568 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:batch_all_reduce: 568 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
155/155 [==============================] - ETA: 0s - loss: 0.0559INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/r

In [17]:
model.load_weights(BEST_PATH)

In [18]:
model.evaluate(val_dataset)

18/18 [==============================] - 1s 59ms/step - loss: 0.0196


0.019620288163423538

0.019411545246839523